# RETFound — Semi-supervised và Few-shot trên Colab

Notebook này nạp `checkpoint-best.pth` từ Google Drive. Semi-supervised phát lại dữ liệu nguồn; few-shot chỉ dùng đúng K ảnh/lớp của một domain đích chưa xuất hiện khi train model nguồn. Chọn GPU runtime trước khi chạy.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Hãy chọn Runtime > Change runtime type > T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone source và cài dependencies

Nếu repository private, hãy clone bằng cơ chế xác thực riêng của bạn; không ghi token trực tiếp vào notebook.

In [ ]:
import os, subprocess, sys
from pathlib import Path

GITHUB_USERNAME = 'Bang334'
GITHUB_REPO = 'dr-diagnostic-system'
GITHUB_BRANCH = 'feat/brset-semi-patient-split'
REPO_DIR = Path('/content') / GITHUB_REPO
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/grading/requirements-train.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/semi_supervised/requirements-research.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle>=2.2.2'], check=True)
source_commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
from ai.semi_supervised.semi_supervised_training import parse_thresholds
threshold_smoke = parse_thresholds('0.93,0.75,0.90,0.80,0.80')
if threshold_smoke != [0.93, 0.75, 0.90, 0.80, 0.80]:
    raise RuntimeError(f'Source không hỗ trợ threshold theo grade: {threshold_smoke}')
print(f'Đã sẵn sàng tại {REPO_DIR}; branch={GITHUB_BRANCH}; commit={source_commit}')
print('✅ Parser threshold theo Grade 0→4 hoạt động đúng.')

## Chọn phương pháp, checkpoint và dữ liệu

- **Checkpoint:** file `checkpoint-best.pth` từ run supervised trên Drive.
- **Labeled replay (Semi):** dataset grading cũ; mặc định tự tải bộ fundus gộp từ Kaggle. Mỗi grade lấy ngẫu nhiên có seed tối đa 2.000 ảnh thật để replay trong mỗi epoch.
- **Unlabeled pool:** mặc định dùng BRSET mirror `kaggle:tanzinabdul/fundus-patientwise-split`. Chỉ split `train` được đưa vào pseudo-label; `validation/test` được giữ ngoài training. Nhãn ICDR của BRSET bị ẩn hoàn toàn. Teacher dùng threshold Grade 0→4 lần lượt `0.93, 0.75, 0.90, 0.80, 0.80`; mỗi grade giữ tối đa 2.000 ảnh confidence cao nhất.
- **Target labeled (Few-shot):** mặc định dùng DeepDRiD v1.1 chính thức (Trung Quốc), chưa nằm trong bộ merged nguồn. Notebook tự tải, chuẩn hóa split và cố định đúng 5 ảnh/lớp từ `train`; `validation` không dùng, `test` chỉ báo cáo trước–sau.

Khi dùng `kaggle:...`, tạo Colab Secret tên `KAGGLE_API_TOKEN`. Không ghi token trực tiếp vào notebook.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

DRIVE_ROOT = Path('/content/drive/MyDrive')
checkpoint_candidates, zip_candidates, named_dirs = [], [], []
for current_root, directories, files in os.walk(DRIVE_ROOT):
    current = Path(current_root)
    if 'checkpoint-best.pth' in files:
        checkpoint_candidates.append(str(current / 'checkpoint-best.pth'))
    zip_candidates.extend(str(current / name) for name in files if name.lower().endswith('.zip'))
    named_dirs.extend(
        str(current / name) for name in directories
        if any(key in name.lower() for key in ('fundus', 'dataset', 'unlabeled', 'target'))
    )
checkpoint_candidates.sort()
zip_candidates.sort()
named_dirs = sorted(set(named_dirs))

DEFAULT_CHECKPOINT_PATH = '/content/drive/MyDrive/retfound_merged_seed42/checkpoint-best.pth'
DEFAULT_LABELED_SOURCE = 'kaggle:sehastrajits/fundus-aptosddridirdeyepacsmessidor'
DEFAULT_UNLABELED_SOURCE = 'kaggle:tanzinabdul/fundus-patientwise-split'
DEFAULT_FEWSHOT_TARGET = 'github:deepdrdoc/DeepDRiD@v1.1'
checkpoint_candidates = list(dict.fromkeys([DEFAULT_CHECKPOINT_PATH] + checkpoint_candidates))
method_widget = widgets.ToggleButtons(
    options=[('Semi-supervised', 'semi'), ('Few-shot', 'fewshot')],
    description='Phương pháp:'
)
checkpoint_widget = widgets.Combobox(
    options=checkpoint_candidates, value=DEFAULT_CHECKPOINT_PATH,
    placeholder='/content/drive/MyDrive/.../checkpoint-best.pth',
    description='Checkpoint:', ensure_option=False, layout=widgets.Layout(width='95%')
)
dataset_widget = widgets.Combobox(
    options=[DEFAULT_LABELED_SOURCE, DEFAULT_FEWSHOT_TARGET] + named_dirs + zip_candidates, value=DEFAULT_LABELED_SOURCE,
    placeholder='kaggle:..., github:owner/repo@tag hoặc thư mục/ZIP',
    description='Labeled replay:', ensure_option=False, layout=widgets.Layout(width='95%')
)
unlabeled_widget = widgets.Combobox(
    options=[DEFAULT_UNLABELED_SOURCE] + named_dirs + zip_candidates, value=DEFAULT_UNLABELED_SOURCE,
    placeholder='kaggle:owner/dataset hoặc thư mục/ZIP ảnh chưa nhãn',
    description='Unlabeled pool:', ensure_option=False, layout=widgets.Layout(width='95%')
)
run_name_widget = widgets.Text(
    value='retfound_semi', description='Tên run cố định:', layout=widgets.Layout(width='70%')
)
method_help = widgets.HTML()
def update_method_fields(change=None):
    if method_widget.value == 'semi':
        dataset_widget.description = 'Labeled replay:'
        if dataset_widget.value.strip() in ('', DEFAULT_FEWSHOT_TARGET):
            dataset_widget.value = DEFAULT_LABELED_SOURCE
        unlabeled_widget.disabled = False
        method_help.value = '<b>Semi:</b> dùng labeled replay cũ + ảnh ngoài chưa nhãn.'
    else:
        dataset_widget.description = 'Target labeled:'
        if dataset_widget.value.strip() in ('', DEFAULT_LABELED_SOURCE):
            dataset_widget.value = DEFAULT_FEWSHOT_TARGET
        unlabeled_widget.disabled = True
        method_help.value = '<b>Few-shot:</b> mặc định DeepDRiD v1.1; notebook tự tải và chuẩn hóa dữ liệu target.'
method_widget.observe(update_method_fields, names='value')
update_method_fields()
display(method_widget, checkpoint_widget, dataset_widget, unlabeled_widget, run_name_widget, method_help)
print(f'Tìm thấy {len(checkpoint_candidates)} best checkpoint trên Drive.')

In [ ]:
import getpass, shutil, zipfile
from google.colab import userdata
from ai.grading.train import find_predefined_splits
from ai.semi_supervised.research_utils import discover_images, load_grading_checkpoint, prepare_deepdrid_target

def get_kaggle_token():
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except Exception:
        token = getpass.getpass('Dán Kaggle API token: ').strip()
    if not token:
        raise RuntimeError('Chưa cung cấp KAGGLE_API_TOKEN')
    os.environ['KAGGLE_API_TOKEN'] = token

def download_kaggle_source(dataset_ref, extract_name):
    target = Path('/content/selected_data') / extract_name
    marker = target / '.kaggle_source'
    if marker.is_file() and marker.read_text(encoding='utf-8').strip() == dataset_ref:
        print(f'Dùng lại dataset Kaggle đã tải: {target}')
        return target.resolve()
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    get_kaggle_token()
    kaggle_cli = [sys.executable, '-m', 'kaggle']
    print(f'Kiểm tra quyền Kaggle: {dataset_ref}')
    subprocess.run(
        kaggle_cli + ['datasets', 'files', '-d', dataset_ref, '--page-size', '20'],
        check=True, capture_output=True, text=True,
    )
    print(f'Đang tải Kaggle dataset {dataset_ref}...')
    subprocess.run(
        kaggle_cli + ['datasets', 'download', '-d', dataset_ref, '-p', str(target)],
        check=True,
    )
    archives = sorted(target.glob('*.zip'))
    if not archives:
        raise FileNotFoundError(f'Kaggle không tạo ZIP trong {target}')
    for archive_path in archives:
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(target)
        archive_path.unlink()
    marker.write_text(dataset_ref, encoding='utf-8')
    return target.resolve()

def download_github_source(repo_ref, extract_name):
    if '@' not in repo_ref or repo_ref.count('/') != 1:
        raise ValueError('GitHub source phải có dạng github:owner/repo@tag')
    repository, git_ref = repo_ref.rsplit('@', 1)
    owner, repo = repository.split('/', 1)
    target = Path('/content/selected_data') / extract_name
    marker = target / '.github_source'
    identity = f'{repository}@{git_ref}'
    if marker.is_file() and marker.read_text(encoding='utf-8').strip() == identity:
        print(f'Dùng lại GitHub dataset đã giải nén: {target}')
        return target.resolve()
    cache_dir = DRIVE_ROOT / 'retfound_datasets'
    cache_dir.mkdir(parents=True, exist_ok=True)
    archive_path = cache_dir / f'{owner}-{repo}-{git_ref}.zip'
    if not archive_path.is_file():
        partial_path = archive_path.with_suffix('.zip.part')
        partial_path.unlink(missing_ok=True)
        url = f'https://github.com/{repository}/archive/refs/tags/{git_ref}.zip'
        print(f'Đang tải {identity} vào Drive cache (khoảng 1.4 GB)...')
        subprocess.run(['curl', '-L', '--fail', '--retry', '3', '--progress-bar', url, '-o', str(partial_path)], check=True)
        partial_path.replace(archive_path)
    else:
        print(f'Dùng lại ZIP trên Drive: {archive_path}')
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    print(f'Đang giải nén {archive_path.name}...')
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(target)
    marker.write_text(identity, encoding='utf-8')
    return target.resolve()

def resolve_source(raw_value, extract_name):
    raw_value = raw_value.strip()
    if not raw_value:
        raise ValueError(f'Chưa chọn nguồn dữ liệu cho {extract_name}')
    if raw_value.lower().startswith('kaggle:'):
        dataset_ref = raw_value.split(':', 1)[1].strip()
        if '/' not in dataset_ref:
            raise ValueError('Kaggle dataset phải có dạng kaggle:owner/dataset')
        return download_kaggle_source(dataset_ref, extract_name)
    if raw_value.lower().startswith('github:'):
        return download_github_source(raw_value.split(':', 1)[1].strip(), extract_name)
    source = Path(raw_value).expanduser()
    if not source.exists():
        raise FileNotFoundError(f'Không tìm thấy: {source}')
    if source.is_dir():
        return source.resolve()
    if source.suffix.lower() != '.zip':
        raise ValueError(f'Chỉ chấp nhận thư mục hoặc ZIP: {source}')
    target = Path('/content/selected_data') / extract_name
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    with zipfile.ZipFile(source) as archive:
        archive.extractall(target)
    return target.resolve()

METHOD = method_widget.value
CHECKPOINT_PATH = Path(checkpoint_widget.value).expanduser().resolve()
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Checkpoint không tồn tại: {CHECKPOINT_PATH}')
# Kiểm tra metadata/kiến trúc trước khi bắt đầu tác vụ dài.
bundle = load_grading_checkpoint(CHECKPOINT_PATH, torch.device('cpu'), require_ce=True)
del bundle
import gc
gc.collect()

if METHOD == 'fewshot' and dataset_widget.value.strip() == DEFAULT_LABELED_SOURCE:
    raise ValueError('Few-shot cần dataset target chưa từng dùng train; không dùng lại bộ merged mặc định')
dataset_role = 'labeled_replay' if METHOD == 'semi' else 'target_labeled_dataset'
dataset_source = resolve_source(dataset_widget.value, dataset_role)
if METHOD == 'fewshot' and dataset_widget.value.strip() == DEFAULT_FEWSHOT_TARGET:
    dataset_source = prepare_deepdrid_target(dataset_source, Path('/content/selected_data/deepdrid_target_split'))
split_dirs = find_predefined_splits(dataset_source)
DATASET_DIR = next(iter(split_dirs.values())).parent.resolve()

UNLABELED_DIR = None
UNLABELED_HOLDOUT_DIRS = {}
if METHOD == 'semi':
    if unlabeled_widget.value.strip() == dataset_widget.value.strip():
        raise ValueError('Dataset có nhãn và nguồn unlabeled không được giống nhau')
    unlabeled_source = resolve_source(unlabeled_widget.value, 'unlabeled_dataset')
    try:
        unlabeled_splits = find_predefined_splits(unlabeled_source)
    except ValueError:
        # Nguồn unlabeled thuần không có cấu trúc train/validation/test.
        UNLABELED_DIR = unlabeled_source.resolve()
    else:
        # Với BRSET patient-wise split, tuyệt đối chỉ pseudo-label split train.
        UNLABELED_DIR = unlabeled_splits['train'].resolve()
        UNLABELED_HOLDOUT_DIRS = {
            name: path.resolve()
            for name, path in unlabeled_splits.items()
            if name != 'train'
        }
    unlabeled_train_paths = discover_images(UNLABELED_DIR)
    if not unlabeled_train_paths:
        raise ValueError(f'Không tìm thấy ảnh trong unlabeled train: {UNLABELED_DIR}')
    print(f'Unlabeled dùng để train: {UNLABELED_DIR} ({len(unlabeled_train_paths):,} ảnh)')
    for split_name, split_path in sorted(UNLABELED_HOLDOUT_DIRS.items()):
        holdout_count = len(discover_images(split_path))
        print(f'🔒 Giữ ngoài semi {split_name}: {split_path} ({holdout_count:,} ảnh)')

OUTPUT_DIR = DRIVE_ROOT / 'retfound_research' / run_name_widget.value / METHOD
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEMI_CONFIG = {
    'epochs': 10, 'patience': 6,
    'batch_size': 2, 'accum_steps': 8,
    'head_lr': '1e-5', 'backbone_lr': '1e-6', 'min_lr': '1e-7',
    # Thứ tự threshold: Grade 0, 1, 2, 3, 4.
    'threshold': '0.93,0.75,0.90,0.80,0.80',
    'max_labeled_per_class': 2000,
    'max_pseudo_per_class': 2000, 'pseudo_weight': '0.25',
    'seed': 42,
}
print(f'Phương pháp : {METHOD}')
print(f'Checkpoint  : {CHECKPOINT_PATH}')
print(f'{"Labeled replay" if METHOD == "semi" else "Target labeled"} : {DATASET_DIR}')
print(f'Unlabeled pool  : {UNLABELED_DIR}')
print(f'Output      : {OUTPUT_DIR}')
if METHOD == 'semi':
    print('Test split được giữ kín và không dùng trong run semi.')
else:
    print('Few-shot chỉ train trên support cố định; test target chỉ báo cáo trước–sau, không chọn model.')

## 🆕 Train mới từ đầu

Cell này bắt đầu semi từ checkpoint grading rồi train trên **labeled replay + pseudo-label mới**. Nếu đã có `checkpoint-last.pth`, cell sẽ dừng và yêu cầu dùng cell **Resume** bên dưới để tránh vô tình ghi đè. Các thông số semi chỉ cần sửa một lần trong `SEMI_CONFIG` ở cell chuẩn bị dữ liệu.

In [ ]:
# TRAIN NEW: không tự động resume
if METHOD == 'semi' and (OUTPUT_DIR / 'checkpoint-last.pth').is_file():
    raise RuntimeError('Run này đã có checkpoint-last.pth. Hãy dùng cell RESUME bên dưới hoặc đổi Tên run.')

if METHOD == 'semi':
    cmd = [
        sys.executable, '-m', 'ai.semi_supervised.semi_supervised_training',
        '--checkpoint', str(CHECKPOINT_PATH),
        '--dataset-dir', str(DATASET_DIR),
        '--unlabeled-dir', str(UNLABELED_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--epochs', str(SEMI_CONFIG['epochs']), '--patience', str(SEMI_CONFIG['patience']),
        '--batch-size', str(SEMI_CONFIG['batch_size']), '--accum-steps', str(SEMI_CONFIG['accum_steps']),
        '--head-lr', SEMI_CONFIG['head_lr'], '--backbone-lr', SEMI_CONFIG['backbone_lr'], '--min-lr', SEMI_CONFIG['min_lr'],
        '--threshold', SEMI_CONFIG['threshold'], '--pseudo-weight', SEMI_CONFIG['pseudo_weight'],
        '--max-labeled-per-class', str(SEMI_CONFIG['max_labeled_per_class']),
        '--max-pseudo-per-class', str(SEMI_CONFIG['max_pseudo_per_class']),
        '--seed', str(SEMI_CONFIG['seed']),
    ]
else:
    cmd = [
        sys.executable, '-m', 'ai.semi_supervised.few_shot_demo',
        '--checkpoint', str(CHECKPOINT_PATH),
        '--target-dataset-dir', str(DATASET_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--epochs', '5', '--train-episodes', '20',
        '--shots', '5', '--queries', '1', '--embedding-dim', '0',
        '--unfreeze-last-blocks', '1', '--forward-batch-size', '2',
        '--encoder-lr', '1e-6', '--projection-lr', '1e-4',
        '--seed', '42',
    ]
def run_live(command):
    command.insert(1, '-u')
    print('Command:', ' '.join(command), flush=True)
    print('=' * 80, flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        print('\n⏹️ Đang dừng tiến trình...', flush=True)
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

run_live(cmd)

## 🔄 Resume semi-supervised từ `checkpoint-last.pth`

Sau khi Colab bị ngắt, chạy lại các cell từ đầu đến hết phần chuẩn bị dữ liệu, rồi chạy **cell này thay cho cell Train mới**. Cell khôi phục model, optimizer, scheduler, AMP scaler, epoch và patience của đúng run `retfound_semi`.

In [ ]:
# RESUME SEMI — tương tự cell resume của notebook grading
# checkpoint-last khôi phục cả model, optimizer, scheduler, AMP scaler và patience.
if METHOD != 'semi':
    raise RuntimeError('Cell này chỉ dùng cho METHOD=semi')

LAST_CHECKPOINT = OUTPUT_DIR / 'checkpoint-last.pth'
if not LAST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Không tìm thấy {LAST_CHECKPOINT}. Hãy chạy cell Train mới trước.')

import torch
resume_state = torch.load(LAST_CHECKPOINT, map_location='cpu', weights_only=False)
completed_epochs = int(resume_state['epoch']) + 1
best_qwk = float(resume_state.get('best_qwk', -1.0))
print(f'✅ Đã hoàn thành {completed_epochs}/{SEMI_CONFIG["epochs"]} epoch; best QWK={best_qwk:.6f}')

if completed_epochs >= SEMI_CONFIG['epochs']:
    print('✅ Run đã đạt tổng số epoch cấu hình; không còn epoch để resume.')
else:
    cmd_resume = [
        sys.executable, '-u', '-m', 'ai.semi_supervised.semi_supervised_training',
        '--checkpoint', str(CHECKPOINT_PATH),
        '--dataset-dir', str(DATASET_DIR),
        '--unlabeled-dir', str(UNLABELED_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--epochs', str(SEMI_CONFIG['epochs']), '--patience', str(SEMI_CONFIG['patience']),
        '--batch-size', str(SEMI_CONFIG['batch_size']), '--accum-steps', str(SEMI_CONFIG['accum_steps']),
        '--head-lr', SEMI_CONFIG['head_lr'], '--backbone-lr', SEMI_CONFIG['backbone_lr'], '--min-lr', SEMI_CONFIG['min_lr'],
        '--threshold', SEMI_CONFIG['threshold'], '--pseudo-weight', SEMI_CONFIG['pseudo_weight'],
        '--max-labeled-per-class', str(SEMI_CONFIG['max_labeled_per_class']),
        '--max-pseudo-per-class', str(SEMI_CONFIG['max_pseudo_per_class']),
        '--seed', str(SEMI_CONFIG['seed']),
        '--resume', str(LAST_CHECKPOINT),
    ]
    print('Command:', ' '.join(cmd_resume), flush=True)
    print('=' * 80, flush=True)
    process = subprocess.Popen(
        cmd_resume, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd_resume)

## Test checkpoint thủ công (có thể chạy sau 3 epoch)

Dừng training cell, chọn `best` để test checkpoint có validation tốt nhất hoặc `last` để test epoch hoàn thành gần nhất, rồi chạy cell bên dưới. Sau khi test, dùng cell **Resume semi-supervised** ở trên để tiếp tục từ `checkpoint-last.pth`. Không nên dùng kết quả test để quyết định tiếp tục/dừng training; hãy dùng validation cho việc đó.

In [ ]:
TEST_CHECKPOINT_KIND = 'best'  # 'best' hoặc 'last'
if METHOD != 'semi':
    raise RuntimeError('Cell test checkpoint này chỉ dùng cho METHOD=semi')
if TEST_CHECKPOINT_KIND not in {'best', 'last'}:
    raise ValueError("TEST_CHECKPOINT_KIND phải là 'best' hoặc 'last'")
test_checkpoint = {
    'best': OUTPUT_DIR / 'checkpoint-best.pth',
    'last': OUTPUT_DIR / 'checkpoint-last.pth',
}[TEST_CHECKPOINT_KIND]
if not test_checkpoint.is_file():
    raise FileNotFoundError(f'Không tìm thấy checkpoint để test: {test_checkpoint}')
print(f'Bắt đầu held-out test với: {test_checkpoint}', flush=True)
test_cmd = [
    sys.executable, '-m', 'ai.semi_supervised.semi_supervised_training',
    '--checkpoint', str(CHECKPOINT_PATH),
    '--dataset-dir', str(DATASET_DIR),
    '--unlabeled-dir', str(UNLABELED_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--batch-size', '2', '--num-workers', '2', '--seed', '42',
    '--resume', str(test_checkpoint), '--eval-only',
]
run_live(test_cmd)

In [ ]:
import json
summary_path = OUTPUT_DIR / 'summary.json'
if not summary_path.is_file():
    raise FileNotFoundError(f'Run chưa tạo summary: {summary_path}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2, ensure_ascii=False))
if METHOD == 'semi':
    if summary.get('test_split_used_for_training') is not False:
        raise RuntimeError('Safety check failed: semi đã dùng test để train')
    if summary.get('test_split_used_for_model_selection') is not False:
        raise RuntimeError('Safety check failed: semi đã dùng test để chọn model')
    if summary.get('test_split_evaluated_after_training') is not True:
        raise RuntimeError('Final held-out test chưa hoàn tất')
    import pandas as pd
    display(pd.read_csv(OUTPUT_DIR / 'pseudo_labels.csv').head(20))
    print((OUTPUT_DIR / 'test_metrics.json').read_text(encoding='utf-8'))
else:
    if summary.get('target_test_used_for_training') is not False or summary.get('target_test_used_for_model_selection') is not False:
        raise RuntimeError('Safety check failed: few-shot đã dùng target test để train/chọn model')
    import pandas as pd
    display(pd.read_csv(OUTPUT_DIR / 'support_manifest.csv'))
    comparison = json.loads((OUTPUT_DIR / 'comparison.json').read_text(encoding='utf-8'))
    print(json.dumps(comparison, indent=2, ensure_ascii=False))

## Sau khi chạy

Few-shot không dùng target validation hay test để chọn checkpoint. `checkpoint-adapted-protonet.pth` lưu sẵn prototype của support nên không cần nạp lại support khi inference, nhưng vẫn là artifact nghiên cứu và chưa tương thích trực tiếp với service grading hiện tại. Chỉ triển khai khi `comparison.json` cho thấy cải thiện ổn định qua nhiều seed/domain.